In [1]:
# ====================================================================
# Developer Utility: Module Auto-Reloading
# --------------------------------------------------------------------
# Uncomment the lines below if you are actively modifying the underlying 
# pi-metaboqc source code. It ensures that changes in .py files are 
# dynamically reloaded without restarting the Jupyter kernel.
# ====================================================================

%load_ext autoreload
%autoreload 2

# ${\pi}$-metaboqc: Interactive Analytical Workflow

This notebook provides an interactive, step-by-step execution orchestration of the `pi-metaboqc` metabolomics data quality control (QC) pipeline. 

By executing each cell sequentially, you can trace the data provenance, inspect intermediate matrices, and visualize quality assessment (QA) diagnostics at each stage of the computational framework.

## Step 00: Environment Initialization
This phase initializes the `pi-metaboqc` computational environment and performs baseline hardware diagnostics. It ensures that the output directory structure is securely mounted before commencing heavy matrix operations.

In [2]:
import os
import pandas as pd
from loguru import logger

import pimqc
import pimqc.io_utils as iu
import pimqc.report_utils as ru
logger.info(f"pimqc.__version__: {pimqc.__version__}")
pimqc.init(check_hardware=False, log_level="INFO", show_progress=True)

from pimqc import (
    build_dataset, MetaboIntAssessor, MetaboIntFilter, MetaboIntCorrector,
    MetaboIntImputer, MetaboIntNormalizer
)

# Define standard directories
DATA_DIR = os.path.join("..", "src", "pimqc", "data")
OUTPUT_DIR = os.path.join(".", "tutorial_output")
iu._check_dir_exists(dir_path=OUTPUT_DIR, handle="makedirs")

# Load pipeline parameters
PARAMS_PATH = os.path.join(DATA_DIR, "pipeline_parameters.toml")
params = iu.load_pipeline_config(config_path=PARAMS_PATH)

# Load raw matrices
meta_df = pd.read_csv(
    os.path.join(DATA_DIR, "project_meta.csv"), header=[0]) 

int_df = pd.read_csv(
    os.path.join(DATA_DIR, "project_intensity.csv"), index_col=[0], header=[0])

2026-05-28 16:36:35.173 | INFO     | __main__:<module>:8 - pimqc.__version__: 1.1.1a1
2026-05-28 16:36:35.178 | SUCCESS  | io_utils:load_pipeline_config:330 - Pipeline configuration successfully loaded and validated via Pydantic.


## Step 01: Raw Dataset Construction
The workflow begins by transforming fragmented raw peak tables and metadata into a standardized `MetaboInt` object. This phase ensures precise coordinate alignment between sample identifiers and feature intensities, establishing a robust structural foundation.

In [3]:
logger.info("Step 01: Dataset Construction...")

step1_dir = os.path.join(OUTPUT_DIR, "01_Raw_Data")
raw_data = build_dataset(
    meta_info=meta_df,
    int_df=int_df,
    pipeline_params=params,
    output_dir=step1_dir
)
is_multi_batch_flag = raw_data.attrs["is_multi_batch"]

2026-05-28 16:36:35.279 | INFO     | __main__:<module>:1 - Step 01: Dataset Construction...
2026-05-28 16:36:35.355 | INFO     | dataset_builder:execute_build:377 - MetaboInt raw dataset saved as: .\tutorial_output\01_Raw_Data\Raw_Data_Intensity.csv
2026-05-28 16:36:35.356 | INFO     | dataset_builder:execute_build:393 - MetaboInt object built: 376 metabolites, 466 samples.
2026-05-28 16:36:35.357 | INFO     | dataset_builder:_audit_dataset_health:246 - Executing dataset health audit...
2026-05-28 16:36:35.358 | INFO     | dataset_builder:_audit_dataset_health:271 - [Audit] No Outlier Reference Features (ORF) detected. This is normal for untargeted datasets; ORF diagnostics will be skipped.


2026-05-28 16:36:36.032 | INFO     | dataset_builder:execute_build:409 - Global acquisition overview plot saved as: .\tutorial_output\01_Raw_Data\Global_Acquisition_Overview.svg


### QA-Step 01: Quality Assessment of Raw Data
Evaluates the baseline data distribution, acquisition sequences, and pooled QC clustering before any computational manipulation occurs.

In [4]:
logger.info("QA-Step 01: Quality Assessment of Raw Data...")

qa_step1_dir = os.path.join(OUTPUT_DIR, "QA_01_Raw_Data")
qa_raw_engine = MetaboIntAssessor(data=raw_data, pipeline_params=params)
qa_raw_engine.execute_assessment(output_dir=qa_step1_dir)

2026-05-28 16:36:36.078 | INFO     | __main__:<module>:1 - QA-Step 01: Quality Assessment of Raw Data...


2026-05-28 16:36:39.535 | INFO     | assessment:execute_assessment:504 - Assessor summary dashboard saved as: .\tutorial_output\QA_01_Raw_Data\QA_Summary_Dashboard.svg
2026-05-28 16:36:39.536 | SUCCESS  | assessment:execute_assessment:505 - Data quality assessment completed.
2026-05-28 16:36:39.537 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_assessment": 00:00:03.456.


## Step 02: Missing Value Classification & Filtering
Implements a topological classification algorithm to segregate missing values into Missing at Random (MAR) and Missing Not at Random (MNAR) based on biological groupings and QC thresholds, strictly filtering out unsalvageable features.

In [5]:
logger.info("Step 02: High Missing Value Feature Filter...")

step2_dir = os.path.join(OUTPUT_DIR, "02_MV_Filtered")
fltr_mv_engine = MetaboIntFilter(data=raw_data, pipeline_params=params)
mv_filter_data = fltr_mv_engine.execute_mv_filtering(output_dir=step2_dir)

2026-05-28 16:36:39.567 | INFO     | __main__:<module>:1 - Step 02: High Missing Value Feature Filter...


2026-05-28 16:36:40.564 | INFO     | filtering:_execute_s1_visualization:486 - High-MV Filter summary dashboard saved as: .\tutorial_output\02_MV_Filtered\MV_Classification_Dashboard.svg
2026-05-28 16:36:40.564 | SUCCESS  | filtering:execute_mv_filtering:393 - High-missing value feature filtering completed.


### QA-Step 02: Quality Assessment of High-MV Filtered Data
Evaluates whether global data distributions remain undisturbed after the removal of high-missing-rate features, ensuring no artificial bias is introduced during topological pruning.

In [6]:
logger.info("QA-Step 02: Quality Assessment of High-MV Filtered Data...")

qa_step2_dir = os.path.join(OUTPUT_DIR,  "QA_02_MV_Filtered")
qa_mv_filter_engine = MetaboIntAssessor(
    data=mv_filter_data, pipeline_params=params)
qa_mv_filter_engine.execute_assessment(output_dir=qa_step2_dir)

2026-05-28 16:36:40.589 | INFO     | __main__:<module>:1 - QA-Step 02: Quality Assessment of High-MV Filtered Data...


2026-05-28 16:36:44.255 | INFO     | assessment:execute_assessment:504 - Assessor summary dashboard saved as: .\tutorial_output\QA_02_MV_Filtered\QA_Summary_Dashboard.svg
2026-05-28 16:36:44.255 | SUCCESS  | assessment:execute_assessment:505 - Data quality assessment completed.
2026-05-28 16:36:44.256 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_assessment": 00:00:03.664.


## Step 03: Signal Drift and Batch Effect Mitigation
Standardizes signal intensities using two selectable strategies:
+ Classical Multi-Stage: Fits intra-batch drift (e.g., QC-RLSC/SVR/RFSC) followed by inter-batch median alignment.
+ Unified Machine Learning: Leverages cross-feature random forests (SERRF) to model and eliminate technical errors in a single step.

In [7]:
logger.info("Step 03: Signal Drift & Batch Effect Correction...")

step3_dir = os.path.join(OUTPUT_DIR, "03_Corrected_Data")
sc_engine = MetaboIntCorrector(data=mv_filter_data, pipeline_params=params)

corrected_stages = sc_engine.execute_signal_correction(output_dir=step3_dir)
final_corr_data = list(corrected_stages.values())[-1]

2026-05-28 16:36:44.291 | INFO     | __main__:<module>:1 - Step 03: Signal Drift & Batch Effect Correction...
2026-05-28 16:36:44.297 | INFO     | correction:execute_signal_correction:969 - AUTO mode enabled. Evaluating multiple methods.
2026-05-28 16:36:44.301 | INFO     | correction:_evaluate_all_methods:806 - --- Evaluating Method: QC-RLSC ---
2026-05-28 16:36:44.302 | INFO     | correction:fit_transform:240 - Phase 1: Executing Intra-batch drift correction with QC-RLSC...
2026-05-28 16:36:48.709 | INFO     | correction:fit_transform:335 - Phase 2: Executing Inter-batch median alignment...
2026-05-28 16:36:49.100 | INFO     | correction:_evaluate_all_methods:907 - QC-RLSC Eval QC RSD: 14.42%
2026-05-28 16:36:49.101 | INFO     | correction:_evaluate_all_methods:806 - --- Evaluating Method: QC-RFSC ---
2026-05-28 16:36:49.101 | INFO     | correction:fit_transform:240 - Phase 1: Executing Intra-batch drift correction with QC-RFSC...


SC [B3]: 100%|███████████████████████████████████████████████████████████████████| 347/347 [Elapsed: 00:14 | ETA: 00:00]


2026-05-28 16:37:55.327 | INFO     | correction:fit_transform:335 - Phase 2: Executing Inter-batch median alignment...
2026-05-28 16:37:55.731 | INFO     | correction:_evaluate_all_methods:907 - QC-RFSC Eval QC RSD: 13.20%
2026-05-28 16:37:55.731 | INFO     | correction:_evaluate_all_methods:806 - --- Evaluating Method: QC-SVR ---
2026-05-28 16:37:55.732 | INFO     | correction:fit_transform:240 - Phase 1: Executing Intra-batch drift correction with QC-SVR...


SC [B3]: 100%|███████████████████████████████████████████████████████████████████| 347/347 [Elapsed: 00:00 | ETA: 00:00]


2026-05-28 16:38:09.901 | INFO     | correction:fit_transform:335 - Phase 2: Executing Inter-batch median alignment...
2026-05-28 16:38:10.293 | INFO     | correction:_evaluate_all_methods:907 - QC-SVR Eval QC RSD: 14.32%
2026-05-28 16:38:10.294 | INFO     | correction:_evaluate_all_methods:806 - --- Evaluating Method: SERRF ---
2026-05-28 16:38:10.294 | INFO     | correction:_prepare_serrf_correlation_matrix:752 - Calculating Spearman correlation on features...
2026-05-28 16:38:10.300 | INFO     | correction:fit_transform:486 - Initializing High-Performance Hybrid SERRF Corrector...


SERRF: 100%|█████████████████████████████████████████████████████████████████████| 347/347 [Elapsed: 00:13 | ETA: 00:00]


2026-05-28 16:38:24.052 | INFO     | correction:_evaluate_all_methods:907 - SERRF Eval QC RSD: 10.00%
2026-05-28 16:38:24.053 | INFO     | correction:_evaluate_all_methods:806 - --- Evaluating Method: RUV ---
2026-05-28 16:38:24.062 | INFO     | correction:_prepare_ruv_control_features:791 - RUV-III Control Features: 20 total (5 predefined, 17 empirical).
2026-05-28 16:38:24.063 | INFO     | correction:fit_transform:548 - Executing Global RUV-III (k=5)...
2026-05-28 16:38:24.066 | WARNING  | correction:fit_transform:563 - NaNs detected. Applying median imputation...
2026-05-28 16:38:24.181 | INFO     | correction:_evaluate_all_methods:907 - RUV Eval QC RSD: 11.09%
2026-05-28 16:38:24.183 | SUCCESS  | correction:execute_signal_correction:997 - Auto selection: SERRF is optimal (Eval QC RSD = 10.00%).
2026-05-28 16:38:24.184 | INFO     | correction:execute_signal_correction:1015 - Assembling evaluation grid for all methods...


2026-05-28 16:38:24.922 | INFO     | correction:execute_signal_correction:1028 - Generating specific RSD plot for: SERRF


2026-05-28 16:38:25.096 | INFO     | correction:execute_signal_correction:1071 - Generating IS plots for SERRF...
2026-05-28 16:38:27.102 | INFO     | correction:execute_signal_correction:1101 - Bypassing IS baseline prediction for SERRF.
2026-05-28 16:38:27.103 | SUCCESS  | correction:execute_signal_correction:1104 - Signal drift correction (SERRF) completed.
2026-05-28 16:38:27.116 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_signal_correction": 00:01:42.820.


### QA-Step 03: Quality Assessment of Corrected Data
Evaluates correction efficacy across both multi-stage and unified strategies. Tracks multi-batch alignment, intra-batch smoothing, and feature-wise RSD reduction to verify technical error suppression. Ensures biological group clustering is driven by intrinsic traits.

In [8]:
qa_step3_dir = os.path.join(OUTPUT_DIR, "QA_03_Signal_Corrected_Data")

qa_engines_dict = {}
for stage_name, stage_data in corrected_stages.items():
    logger.info(f"Executing Quality Assessment (QA) for: {stage_name}")
    qa_corr_engine = MetaboIntAssessor(data=stage_data, pipeline_params=params) 
    qa_corr_engine.execute_assessment(
        output_dir=os.path.join(qa_step3_dir, stage_name))
    qa_engines_dict[stage_name] = qa_corr_engine

2026-05-28 16:38:27.142 | INFO     | __main__:<module>:5 - Executing Quality Assessment (QA) for: Global SERRF


2026-05-28 16:38:30.753 | INFO     | assessment:execute_assessment:504 - Assessor summary dashboard saved as: .\tutorial_output\QA_03_Signal_Corrected_Data\Global SERRF\QA_Summary_Dashboard.svg
2026-05-28 16:38:30.753 | SUCCESS  | assessment:execute_assessment:505 - Data quality assessment completed.
2026-05-28 16:38:30.754 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_assessment": 00:00:03.609.


## Step 04: Low-Quality Feature Filtering
Performs a rigorous reproducibility check using the analytical variance (RSD) of Pooled QC samples and Blank/QC ratio. Features exhibiting unacceptable technical variance post-correction are permanently eliminated from the quantitative matrix.

In [9]:
logger.info("Step 04: Low-Quality Feature Filtering...")

step4_dir = os.path.join(OUTPUT_DIR, "04_Quality_Filtered")
fltr_low_quality_engine = MetaboIntFilter(
    data=final_corr_data, pipeline_params=params)
low_quality_filter_data = fltr_low_quality_engine.execute_quality_filtering(
    output_dir=step4_dir)

2026-05-28 16:38:30.786 | INFO     | __main__:<module>:1 - Step 04: Low-Quality Feature Filtering...
2026-05-28 16:38:30.790 | INFO     | filtering:execute_quality_filtering:629 - Features before filtering: 347
2026-05-28 16:38:30.797 | INFO     | filtering:execute_quality_filtering:649 - Features after Blank/QC check: 261
2026-05-28 16:38:30.805 | INFO     | filtering:execute_quality_filtering:670 - Features after QC RSD check: 248
2026-05-28 16:38:30.885 | INFO     | filtering:execute_quality_filtering:704 - Data after low-quality features filtering saved as: .\tutorial_output\04_Quality_Filtered\Filtered_Data_Low-quality_Features.csv


2026-05-28 16:38:31.643 | INFO     | filtering:_execute_s2_visualization:792 - Low-quality Filter summary dashboard saved as: .\tutorial_output\04_Quality_Filtered\Low-quality_Filtering_Dashboard.svg
2026-05-28 16:38:31.644 | SUCCESS  | filtering:execute_quality_filtering:715 - Low-quality features filtering completed.


### QA-Step 04: Quality Assessment on Low-Quality Feature Filtered Data
A pre-imputation health check is performed on the refined dataset. This evaluation confirms that the surviving features represent high-fidelity biological signals, ensuring the matrix is optimally prepared for missing value reconstruction

In [10]:
logger.info(
    "QA-Step 04: Quality Assessment on Low-Quality Feature Filtered Data...")

qa_step4_dir = os.path.join(OUTPUT_DIR, "QA_04_Quality_Filtered")
qa_low_quality_filter_engine = MetaboIntAssessor(
    data=low_quality_filter_data, pipeline_params=params)
qa_low_quality_filter_engine.execute_assessment(output_dir=qa_step4_dir)

2026-05-28 16:38:31.671 | INFO     | __main__:<module>:1 - QA-Step 04: Quality Assessment on Low-Quality Feature Filtered Data...


2026-05-28 16:38:35.136 | INFO     | assessment:execute_assessment:504 - Assessor summary dashboard saved as: .\tutorial_output\QA_04_Quality_Filtered\QA_Summary_Dashboard.svg
2026-05-28 16:38:35.136 | SUCCESS  | assessment:execute_assessment:505 - Data quality assessment completed.
2026-05-28 16:38:35.137 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_assessment": 00:00:03.463.


## Step 05: Missing Value Imputation
An autonomous multi-algorithm benchmarking simulation is executed for MAR. The optimal algorithm is programmatically selected based on its ability to reconstruct established distributions while minimizing bias in the original variance structure.

Execute min-value imputation for MNAR.

In [11]:
logger.info("Step 05: Missing Value Imputation...")

step5_dir = os.path.join(OUTPUT_DIR, "05_Imputation")
imp_engine = MetaboIntImputer(
    data=low_quality_filter_data, pipeline_params=params)
imputed_data = imp_engine.execute_imputation(output_dir=step5_dir)

2026-05-28 16:38:35.172 | INFO     | __main__:<module>:1 - Step 05: Missing Value Imputation...
2026-05-28 16:38:35.178 | INFO     | imputation:execute_imputation:759 - Hybrid Imputation Engine Initialized. MAR: Auto (Evaluating KNN=5, LLS (K=15), MinProb, Median) | MNAR: QRILC | Sim_Mask: 0.05
2026-05-28 16:38:35.187 | INFO     | imputation:execute_imputation:773 - Applying QRILC to 3 MNAR features.
2026-05-28 16:38:35.327 | INFO     | imputation:select_best_algorithm:642 - Simulating "KNN" on MAR subset...
2026-05-28 16:38:37.694 | INFO     | imputation:select_best_algorithm:642 - Simulating "MinProb" on MAR subset...
2026-05-28 16:38:38.136 | INFO     | imputation:select_best_algorithm:642 - Simulating "Median" on MAR subset...
2026-05-28 16:38:38.468 | INFO     | imputation:select_best_algorithm:642 - Simulating "LLS" on MAR subset...
2026-05-28 16:38:38.630 | INFO     | imputation:select_best_algorithm:653 - Optimal MAR algorithm selected: KNN
2026-05-28 16:38:38.631 | INFO     | 

2026-05-28 16:38:41.932 | INFO     | imputation:execute_imputation:891 - Imputer summary dashboard saved as: .\tutorial_output\05_Imputation\Imputation_Dashboard_KNN.svg
2026-05-28 16:38:41.932 | SUCCESS  | imputation:execute_imputation:894 - Missing value imputation completed successfully.
2026-05-28 16:38:41.934 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_imputation": 00:00:06.759.


### QA-Step 05: Quality Assessment on Imputed Data
Evaluates the extent to which synthetic data points might introduce artificial clustering or distort natural biological correlations, with a specific focus on the stability of low-abundance signals.

In [12]:
logger.info("QA-Step 05: Quality Assessment of Imputated Data...")

qa_step5_dir = os.path.join(OUTPUT_DIR, "QA_05_Imputed_Data")
qa_imp_engine = MetaboIntAssessor(data=imputed_data, pipeline_params=params)
qa_imp_engine.execute_assessment(output_dir=qa_step5_dir)

2026-05-28 16:38:41.958 | INFO     | __main__:<module>:1 - QA-Step 05: Quality Assessment of Imputated Data...


2026-05-28 16:38:45.256 | INFO     | assessment:execute_assessment:504 - Assessor summary dashboard saved as: .\tutorial_output\QA_05_Imputed_Data\QA_Summary_Dashboard.svg
2026-05-28 16:38:45.256 | SUCCESS  | assessment:execute_assessment:505 - Data quality assessment completed.
2026-05-28 16:38:45.257 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_assessment": 00:00:03.296.


## Step 06: Data Normalization
Applies advanced normalization techniques (e.g., VSN, Quantile) to stabilize heteroscedastic variance and harmonize global intensity scales across all biological samples.

In [13]:
logger.info("Step 06: Data Normalization...")

step6_dir = os.path.join(OUTPUT_DIR, "06_Normalized_Data")
norm_engine = MetaboIntNormalizer(imputed_data, pipeline_params=params)
normalized_data = norm_engine.execute_normalization(
    output_dir=step6_dir)

2026-05-28 16:38:45.289 | INFO     | __main__:<module>:1 - Step 06: Data Normalization...
2026-05-28 16:38:45.292 | INFO     | normalization:execute_normalization:670 - Permanently dropping 24 Blank samples.
2026-05-28 16:38:45.293 | INFO     | normalization:execute_normalization:672 - Applying Normalization | Method: VSN | Log: False
2026-05-28 16:39:07.961 | INFO     | normalization:execute_normalization:694 - Calculating normalization-related metrics...
2026-05-28 16:39:10.192 | INFO     | normalization:execute_normalization:701 - Generating diagnostic plots for normalization...


2026-05-28 16:39:13.389 | INFO     | normalization:execute_normalization:711 - Normalization summary dashboard saved as: .\tutorial_output\06_Normalized_Data\Normalization_Dashboard_VSN.svg
2026-05-28 16:39:13.389 | SUCCESS  | normalization:execute_normalization:712 - Data normalization completed successfully.
2026-05-28 16:39:13.390 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_normalization": 00:00:28.099.


### QA-Step 06: Quality Assessment on Normalized Data
Monitors spatial distribution fidelity using Jensen-Shannon Divergence (JSD) and Wasserstein metrics, evaluating whether the normalization successfully mitigated systematic biases without obliterating genuine biological differences.

In [14]:
logger.info("QA-Step 06: Quality Assessment of Normalized Data...")

qa_step6_dir = os.path.join(OUTPUT_DIR, "QA_06_Norm_Data")
qa_norm_engine = MetaboIntAssessor(
    data=normalized_data, pipeline_params=params)
qa_norm_engine.execute_assessment(output_dir=qa_step6_dir)

2026-05-28 16:39:13.420 | INFO     | __main__:<module>:1 - QA-Step 06: Quality Assessment of Normalized Data...


2026-05-28 16:39:16.790 | INFO     | assessment:execute_assessment:504 - Assessor summary dashboard saved as: .\tutorial_output\QA_06_Norm_Data\QA_Summary_Dashboard.svg
2026-05-28 16:39:16.791 | SUCCESS  | assessment:execute_assessment:505 - Data quality assessment completed.
2026-05-28 16:39:16.791 | SUCCESS  | io_utils:time_wrap:391 - Execution time of "execute_assessment": 00:00:03.368.


## Step 07: Sequential Audit Report Compilation
Assembles all intermediate vector graphics, mathematical metrics, and tracking logs to autonomously generate a comprehensive, human-readable HTML/PDF and Markdown audit report.

In [15]:
logger.info("Step 07: Sequential Audit Report Compilation...")

REPORT_DIR = "07_Report_Summary"
# 1. Process Visual Assets 
visual_rep = ru.VisualAssetReporter(base_dir=OUTPUT_DIR)
visual_rep.compile_assessor_report(
    report_folder=REPORT_DIR, is_multi_batch=is_multi_batch_flag
)

# 2. Dynamic QA Mapping
qa_corr_metrics = {}
stage_key_map = {
    "Intra-batch corrected": "intra_batch_correction",
    "Inter-batch corrected": "inter_batch_correction",
    "Global SERRF": "global_correction"
}

for stage_name, engine in qa_engines_dict.items():
    safe_key = stage_key_map.get(stage_name, "unknown_correction")
    qa_corr_metrics[safe_key] = engine.assessment_metrics

# 3. Assemble Final Metrics
pipeline_metrics_objs = {
    "raw_dataset": raw_data.dataset_metrics,
    "high_mv_feature_filtering": mv_filter_data.mv_filtering_metrics,
    "signal_correction": final_corr_data.correction_metrics, 
    "low_quality_feature_filtering": 
        low_quality_filter_data.quality_filtering_metrics,
    "missing_value_imputation": imputed_data.imputation_metrics,
    "normalization": normalized_data.normalization_metrics
}

qa_metrics_objs = {
    "raw_dataset": qa_raw_engine.assessment_metrics,
    "high_mv_feature_filtering": qa_mv_filter_engine.assessment_metrics,
    **qa_corr_metrics, 
    "low_quality_feature_filtering": 
        qa_low_quality_filter_engine.assessment_metrics, 
    "missing_value_imputation": qa_imp_engine.assessment_metrics,
    "normalization": qa_norm_engine.assessment_metrics
}

# 4. Initialize reporter and generate ONE markdown file
print(f"Initializing narrative reporter at workspace: {OUTPUT_DIR}")
md_reporter = ru.NarrativeStatsReporter(base_dir=OUTPUT_DIR)

md_reporter.generate_markdown(
    pipeline_metrics=pipeline_metrics_objs, 
    qa_metrics=qa_metrics_objs,
    report_folder=REPORT_DIR
)

success = md_reporter.export_report(pdf_engine="weasyprint")

if success:
    logger.success("PI-METABOQC PIPELINE COMPLETED SUCCESSFULLY.")

2026-05-28 16:39:16.827 | INFO     | __main__:<module>:1 - Step 07: Sequential Audit Report Compilation...
2026-05-28 16:39:16.828 | INFO     | report_utils:compile_assessor_report:227 - Multi-batch design detected. Assembling Batch Grid.


2026-05-28 16:39:16.908 | SUCCESS  | report_utils:compile_assessor_report:268 - Report SVG assets compiled at: tutorial_output\07_Report_Summary\assets
Initializing narrative reporter at workspace: .\tutorial_output
2026-05-28 16:39:17.036 | INFO     | report_utils:generate_markdown:1066 - Generating COMPREHENSIVE narrative report...
2026-05-28 16:39:17.069 | SUCCESS  | report_utils:generate_markdown:1076 - Comprehensive report generated: tutorial_output\07_Report_Summary\Report_Comprehensive.md
2026-05-28 16:39:17.069 | INFO     | report_utils:generate_markdown:1066 - Generating BRIEF narrative report...
2026-05-28 16:39:17.078 | SUCCESS  | report_utils:generate_markdown:1076 - Brief report generated: tutorial_output\07_Report_Summary\Report_Brief.md
2026-05-28 16:39:17.205 | INFO     | report_utils:export_report:1221 - --- Exporting PDF for: Report_Comprehensive.md ---
2026-05-28 16:39:17.206 | INFO     | report_utils:_render_weasyprint:1148 - Attempting PDF export via WeasyPrint...


In [16]:
# md_reporter.consolidate_metrics(pipeline_metrics=pipeline_metrics_objs, 
#     qa_metrics=qa_metrics_objs)